# Retrieval Agent With Custom Tools

In this notebook, I build a retrieval agent using the Hugging Face `smolagents` library.

The agent answers questions about the guests attending a party by searching through a small knowledge base that I store directly in the notebook. This is a simple form of retrieval: instead of the model guessing an answer, the agent calls a tool that looks up the real information first.

In this notebook, I will learn how to:

- Store a small knowledge base as plain Python data
- Write a keyword based retriever without using any embedding model
- Turn the retriever into an agent tool using the `@tool` decorator
- Rebuild the same tool as a class that inherits from `Tool`
- Combine the retriever with a web search tool so the agent can fall back to the internet
- Let the agent decide which tool to use for each question

I keep everything lightweight on purpose. There is no vector database and no model running on my machine, so the notebook stays fast to run.

## 1. Importing Libraries and Creating the Model

First I import the pieces I need from `smolagents`.

`CodeAgent` is the agent that writes and runs Python code to solve a task. `tool` and `Tool` are the two ways of creating a custom tool. `InferenceClientModel` is the language model, which runs on the Hugging Face Inference API rather than on my own machine.

In [ ]:
from smolagents import (
    CodeAgent,
    DuckDuckGoSearchTool,
    InferenceClientModel,
    Tool,
    tool
)

model = InferenceClientModel(
    model_id="Qwen/Qwen2.5-Coder-32B-Instruct"
)

## 2. Building the Guest Knowledge Base

The knowledge base is the information the agent is allowed to look things up in. In a real project this would be a database or a set of documents, but here a list of dictionaries is enough.

Each guest has a name, a relation, a short description and the year they were born. The language model has no way of knowing any of this, so the only way the agent can answer correctly is by calling my retriever tool.

In [ ]:
guests = [
    {
        "name": "Ada Lovelace",
        "relation": "best friend",
        "description": "A respected mathematician and writer, known for her work on Charles Babbage's Analytical Engine. She is often described as the first computer programmer.",
        "born": 1815,
    },
    {
        "name": "Alan Turing",
        "relation": "old university contact",
        "description": "A mathematician and logician who formalised the idea of computation. He enjoys long walks and does not enjoy small talk.",
        "born": 1912,
    },
    {
        "name": "Grace Hopper",
        "relation": "colleague from the navy",
        "description": "A computer scientist and rear admiral who worked on the first compilers. She prefers her coffee black and her meetings short.",
        "born": 1906,
    },
    {
        "name": "Marie Curie",
        "relation": "distant relative",
        "description": "A physicist and chemist who carried out pioneering research on radioactivity. She is the only person to win a Nobel Prize in two different sciences.",
        "born": 1867,
    },
]

print(f"The knowledge base contains {len(guests)} guests.")

## 3. Turning a Guest Record Into Readable Text

A tool has to return text, because the text is what the language model actually reads.

So before writing the retriever I write a small helper that turns one guest dictionary into a clean, readable paragraph. Keeping this in its own function means the retriever stays short and I can change the wording in one place.

In [ ]:
def format_guest(guest: dict) -> str:
    """Turns a single guest record into a readable block of text."""
    return (
        f"Name: {guest['name']}\n"
        f"Relation: {guest['relation']}\n"
        f"Born: {guest['born']}\n"
        f"Description: {guest['description']}"
    )


print(format_guest(guests[0]))

## 4. Writing the Keyword Scoring Function

Now I write the part that actually decides which guest matches a question.

Real retrieval systems compare embeddings, but that means loading a model and storing vectors. For a knowledge base this small, counting shared words works well and costs nothing.

The function lowercases the query and the record, splits both into words, and counts how many words they have in common. A guest whose text shares more words with the question gets a higher score.

In [ ]:
import re


def tokenize(text: str) -> set:
    """Splits text into a set of lowercase words."""
    return set(re.findall(r"[a-z0-9]+", text.lower()))


def score_guest(query: str, guest: dict) -> int:
    """Counts how many words the query and the guest record share."""
    query_words = tokenize(query)
    guest_words = tokenize(format_guest(guest))
    return len(query_words & guest_words)


for guest in guests:
    print(guest["name"], "->", score_guest("Tell me about the mathematician Ada", guest))

## 5. Ranking the Guests

With a score for each guest, retrieval is just sorting.

I sort every guest by score, drop the ones that share no words at all with the question, and keep the best few. Returning more than one result is useful because the agent can read them and pick the one that really answers the question.

In [ ]:
def search_guests(query: str, top_k: int = 2) -> list:
    """Returns the guests whose records best match the query."""
    scored = [(score_guest(query, guest), guest) for guest in guests]
    matches = [(score, guest) for score, guest in scored if score > 0]
    matches.sort(key=lambda pair: pair[0], reverse=True)
    return [guest for score, guest in matches[:top_k]]


for guest in search_guests("Who worked on compilers?"):
    print(guest["name"])

## 6. Improving the Scores by Ignoring Common Words

Testing the scoring function showed a problem. Words like "the", "and" or "a" appear in almost every record, so a guest can pick up points from a question that has nothing to do with them.

The fix is a stopword list. I remove these very common words before comparing, so only the meaningful words count. After this change the scores are lower but much more honest.

In [ ]:
STOPWORDS = {
    "a", "an", "and", "the", "is", "was", "are", "were", "of", "on", "in",
    "to", "for", "with", "who", "what", "which", "me", "my", "about",
    "tell", "her", "his", "she", "he", "it", "at", "as", "by", "that",
}


def tokenize(text: str) -> set:
    """Splits text into a set of lowercase words, ignoring common words."""
    words = set(re.findall(r"[a-z0-9]+", text.lower()))
    return words - STOPWORDS


for guest in guests:
    print(guest["name"], "->", score_guest("Tell me about the mathematician Ada", guest))

## 7. Turning the Retriever Into a Tool

The retriever works, but the agent cannot use a plain Python function. It needs a tool.

The `@tool` decorator does the conversion for me. The important detail is the docstring: `smolagents` reads it to build the tool description that the model sees, so the description and the argument list have to explain clearly when this tool should be used.

In [ ]:
@tool
def guest_info_tool(query: str) -> str:
    """
    Looks up information about the guests invited to the party.

    Args:
        query (str): A name or a few words describing the guest to look up.
    """
    matches = search_guests(query)

    if not matches:
        return "No guest matching that description is on the invitation list."

    return "\n\n".join(format_guest(guest) for guest in matches)

## 8. Testing the Tool on Its Own

Before giving the tool to an agent I call it directly.

This is worth doing every time. If the tool returns something wrong here, the agent will only make the problem harder to see, because then I cannot tell whether the mistake came from my code or from the model.

In [ ]:
print(guest_info_tool("Who is Ada Lovelace?"))
print()
print(guest_info_tool("Tell me about the guest who studied radioactivity"))
print()
print(guest_info_tool("Is Napoleon coming?"))

## 9. Creating the Retrieval Agent

Now I give the tool to a `CodeAgent`.

The agent receives the question, decides that it needs guest information, writes a line of Python that calls `guest_info_tool`, reads the result and then writes the final answer in its own words.

In [ ]:
guest_agent = CodeAgent(
    tools=[guest_info_tool],
    model=model
)